In [1]:
# imports

import os  
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from io import BytesIO
import wave
from google import genai
from google.genai import types

/Users/omkardabholkar/my_llm_projects/DeutschMate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialization

load_dotenv(override=True)#reads the env file and if some passwords already exists, throw it out and load the new one

gemini_api_key = os.getenv('GEMINI_API_KEY')#pulls the key out of environment and stores in variable
if gemini_api_key: 
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")

MODEL = "gemini-3.5-flash-lite"               # chat model, via the OpenAI-compatible endpoint (fast, no thinking by default)
TTS_MODEL = "gemini-3.1-flash-tts-preview"    # native SDK, pronunciation audio (still preview)


client = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Gemini's own SDK - still needed for text-to-speech
genai_client = genai.Client(api_key=gemini_api_key)

DB = "vocab.db"

Gemini API Key exists and begins AQ.Ab8RN


In [3]:
def init_vocab_db():
    with sqlite3.connect(DB) as conn:#opens the vocab.db file as a database connection, with block gurantees that the connection closes properly even if something errors
        cursor = conn.cursor()#cursor is the object through which we can use sql commands through the connection
        cursor.execute(''' 
            CREATE TABLE IF NOT EXISTS vocab (
                german TEXT PRIMARY KEY,
                english TEXT,                                   
                example_sentence TEXT,
                times_reviewed INTEGER DEFAULT 0
            )
        ''')
        conn.commit()#creates a table with 4 columns , commit saves changes to the database

init_vocab_db()

In [4]:
def save_new_word(german, english, example_sentence):
    print(f"DATABASE TOOL CALLED: Saving '{german}' -> '{english}'", flush=True)#for debugging purposes that it used the tool
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO vocab (german, english, example_sentence, times_reviewed)
            VALUES (?, ?, ?, 0)
            ON CONFLICT(german) DO UPDATE SET times_reviewed = times_reviewed + 1
        ''', (german.lower(), english, example_sentence))
        conn.commit()
    return f"Saved '{german}' ({english}) to the vocab list."


def get_known_words():
    print("DATABASE TOOL CALLED: Fetching known vocab", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT german, english, times_reviewed FROM vocab ORDER BY times_reviewed DESC')
        rows = cursor.fetchall()
    if not rows:
        return "The student hasn't saved any vocabulary yet."
    return "; ".join([f"{g} = {e} (reviewed {t}x)" for g, e, t in rows])

In [5]:
# quick manual test, same idea as get_ticket_price("Paris") in Day 5
print(save_new_word("Kuehlschrank", "refrigerator", "Der Kuehlschrank ist leer."))
print(get_known_words())

DATABASE TOOL CALLED: Saving 'Kuehlschrank' -> 'refrigerator'
Saved 'Kuehlschrank' (refrigerator) to the vocab list.
DATABASE TOOL CALLED: Fetching known vocab
kuehlschrank = refrigerator (reviewed 2x); ich bin neu in deutschland = I am new in Germany (reviewed 0x); herzlich willkommen = welcome (reviewed 0x); ich bin neu in deutschland. = I am new in Germany (reviewed 0x)


In [6]:
save_word_function = {  #tells llm here is the tool you are allowed to call
    "name": "save_new_word",
    "description": "Save a new German word the student just learned, with its English translation and an example sentence, to their personal vocab list. Call this at most once or twice per reply, only for the most important new word(s).",# explain when and why ai can use this func
    "parameters": {
        "type": "object",
        "properties": {
            "german": {
                "type": "string",
                "description": "The German word or short phrase being taught",
            },
            "english": {
                "type": "string",
                "description": "Its English translation",
            },
            "example_sentence": {
                "type": "string",
                "description": "A short, simple example sentence in German using the word",
            },
        },
        "required": ["german", "english", "example_sentence"],
        "additionalProperties": False
    }
}

get_known_words_function = {
    "name": "get_known_words",
    "description": "Get the list of German words the student has already saved, with how many times each has been reviewed. Call this if the student asks what they've learned so far, or to decide what to review vs teach next.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": save_word_function},
    {"type": "function", "function": get_known_words_function},
]
tools

[{'type': 'function',
  'function': {'name': 'save_new_word',
   'description': 'Save a new German word the student just learned, with its English translation and an example sentence, to their personal vocab list. Call this at most once or twice per reply, only for the most important new word(s).',
   'parameters': {'type': 'object',
    'properties': {'german': {'type': 'string',
      'description': 'The German word or short phrase being taught'},
     'english': {'type': 'string', 'description': 'Its English translation'},
     'example_sentence': {'type': 'string',
      'description': 'A short, simple example sentence in German using the word'}},
    'required': ['german', 'english', 'example_sentence'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_known_words',
   'description': "Get the list of German words the student has already saved, with how many times each has been reviewed. Call this if the student asks what they've learned so fa

In [7]:
system_message = """
You are Anna, a friendly German language companion for an A1-level student who wants
to keep improving through natural conversation (not grammar drills).

Rules:
- Write mostly in German at roughly A2 level; for anything trickier, add a short
  English gloss in [brackets] right after it.
- If the student makes a mistake, gently show the corrected sentence first, then a
  one-line reason in English, then keep the conversation going.
- At most once or twice per reply, if you teach or use an important new German word
  or phrase for the first time, call the save_new_word tool for it. Do this quietly,
  in the background - never mention the tool itself to the student.
- If the student asks what they know so far, or what to review, call get_known_words.
- Keep replies short: 2-4 sentences, so it stays a back-and-forth conversation.
"""

In [8]:
def handle_tool_calls(message):#message contains llm's request for tools
    responses = []#we'll put responses from tools in here.
    for tool_call in message.tool_calls: #since we can have have multiple tool requests
        arguments = json.loads(tool_call.function.arguments)#converts the json request from llm to a python readable dictionary
        if tool_call.function.name == "save_new_word": 
            result = save_new_word(**arguments)#unpacks the dictionary gives the arguments as the func expects
        elif tool_call.function.name == "get_known_words":
            result = get_known_words()
        else:
            result = "Unknown tool"
        responses.append({ #append it to the empty list we created
            "role": "tool",   #its a tool's answer
            "content": result, #actual answer from tool. 
            "tool_call_id": tool_call.id
        })
    return responses #back to llm returned

In [9]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]#system prompt+previous history(hallo+assistant response)+ current user message
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls": #as long as llm is asking for tools, keep executing
        tool_message = response.choices[0].message #contains llm tool request
        responses = handle_tool_calls(tool_message) #python executes the database operation
        messages.append(tool_message)#add tool request to conversation
        messages.extend(responses)#add tool results.
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)#after receiving the tool result, ask llm to give final response to the user

    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
def talker(german_text):
    response = genai_client.models.generate_content(
        model=TTS_MODEL,    #text to speech model call
        contents=german_text,   #generate audio for this german text
        config=types.GenerateContentConfig(    #request audio  
            response_modalities=["AUDIO"],#audio as output needed
            speech_config=types.SpeechConfig(   #voice configuration
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
                )
            ),
        ),
    )
    pcm_data = response.candidates[0].content.parts[0].inline_data.data #extracts audio

    # Gemini TTS returns raw PCM (24kHz, 16-bit, mono) - wrap it into a playable WAV.
    buffer = BytesIO() #creates an in memory binary buffer
    with wave.open(buffer, "wb") as wf: #open a wav file in memory for writing wb-write binary
        wf.setnchannels(1)#mono channel
        wf.setsampwidth(2)#2 bytes 
        wf.setframerate(24000)#24000 samples per second
        wf.writeframes(pcm_data)#the raw audio is placed inside a wav container
    buffer.seek(0)
    return buffer.read()#returns the finished wav bytes

In [11]:
def chat_multimodal(history):   #combine chat + tool +voice
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history    #new user because ui callback has already inserted user message to history
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools) #chat + tool calling

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        responses = handle_tool_calls(tool_message)
        messages.append(tool_message)
        messages.extend(responses)
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content #get the answer given by llm to user after receiving response from tool
    history += [{"role": "assistant", "content": reply}]#add that to history 

    voice = talker(reply)   #send the reply of llm to user to talker function so it reads that out loud, returns wav bytes

    return history, voice #so the ui receives updated chat + audio

In [ ]:
# Callbacks (along with the chat_multimodal() function above)

def put_message_in_chatbot(message, history): #just ui helper function
    return "", history + [{"role": "user", "content": message}] #after pressing enter it removes the text from the text box and also adds it to the history (user message)

# UI definition

with gr.Blocks() as ui: #building gradio interface
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)#automatically play pronounciation when arrives
    with gr.Row():
        message = gr.Textbox(label="Sprich mit deinem Deutsch-Coach (chat in German or English):")#creates the user input box

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat_multimodal, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Saving 'schlecht' -> ' schlecht'
DATABASE TOOL CALLED: Saving 'der Alltag' -> 'everyday life'
DATABASE TOOL CALLED: Saving 'der Alltag' -> 'everyday life'
DATABASE TOOL CALLED: Fetching known vocab


# Exercises and ideas to extend this

- **Spaced repetition**: use `times_reviewed` (already in the DB) to have the tutor deliberately re-use older, less-reviewed words instead of only new ones.
- **A daily quiz mode**: a second tool, `quiz_me`, that pulls a random saved word from `get_known_words` and asks the student to use it in a sentence.
- **Difficulty ramp**: add a `level` column to `vocab.db` and let the system prompt reference it, so the tutor can nudge from A2 toward B1 over time.
- **Bringing image generation back**: `client.images.generate(model=..., prompt=..., response_format="b64_json")` works on the OpenAI-compatible endpoint once billing is enabled on your Google AI Studio project (image models have a 0 free-tier quota).